Downloading the requirements for the dataset download

In [1]:
!pip install sentence-transformers datasets -q

In [3]:
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings; warnings.filterwarnings('ignore')


Loading the Simple English Wikipedia Dataset
### Problem 3: Wikipedia Dataset Loading Issue

The older `wikipedia` dataset required `trust_remote_code=True`, but this is now blocked because dataset scripts have been deprecated.

To resolve this, we switch to the updated and officially supported dataset:

**`wikimedia/wikipedia`**

For Simple English Wikipedia, the correct snapshot to use is:

**`20231101.simple`**

This version does not rely on deprecated scripts and works smoothly with the current Hugging Face dataset system.

In [6]:
print('Simple English Wikipedia Dataset')
wiki = load_dataset('wikimedia/wikipedia', '20231101.simple', split='train')
wiki_subset = wiki.select(range(2000))

Simple English Wikipedia Dataset


README.md: 0.00B [00:00, ?B/s]

20231101.simple/train-00000-of-00001.par(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/241787 [00:00<?, ? examples/s]

Using full article text

In [7]:
documents = [ex['text'] for ex in wiki_subset]
titles = [ex['title'] for ex in wiki_subset]

In [8]:
print(f'Loaded {len(documents)} Wikipedia articles')
print(f'Sample title: {titles[0]}')
print(f'Sample text: {documents[0][:200]}')

Loaded 2000 Wikipedia articles
Sample title: April
Sample text: April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of the four months to have 30 days.

April always begins on the same day 


In [9]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
doc_inputs = [f"{title}. {text[:400]}" for title, text in zip(titles, documents)]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
embeddings = model.encode(doc_inputs, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
sim_matrix = cosine_similarity(embeddings)
np.fill_diagonal(sim_matrix, 0)
print(f'Sim matrix shape: {sim_matrix.shape}')

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Sim matrix shape: (2000, 2000)


Computing the Retrieval Function

Retrieve the top-k most relevant documents for a natural language query.

Arguments:
- query (str): Natural language query.
- documents (list of strings): Document corpus.
- embeddings (numpy array): Precomputed embeddings (N, D).
- model (SentenceTransformer): Model used to encode the query.
- top_k (int): Number of documents to retrieve.
- titles (list of strings, optional): Titles for display.

Returns:
- A list of dictionaries containing the top-k results sorted by similarity (highest first).

In [16]:
def retrieve(query, documents, embeddings, model, top_k=5, titles=None):
    # (i) Compute query embedding
    query_emb = model.encode([query])[0]

    # (ii) Compute cosine similarity between query and all docs
    scores = cosine_similarity([query_emb], embeddings)[0]

    # (iii) Sort documents by similarity (descending)
    ranked_indices = np.argsort(scores)[::-1]

    # (iv) Return top-k most similar documents
    results = []
    for idx in ranked_indices[:top_k]:
        result = {
            'rank': len(results) + 1,
            'index': int(idx),
            'score': float(scores[idx]),
            'text': str(documents[idx]),        # ← str() wrap
        }
        if titles is not None:
            result['title'] = str(titles[idx])  # ← str() wrap
        results.append(result)
    assert len(results) > 0, "retrieve() returned empty list — check embeddings/documents."
    return results

Test Retrieval System by using sample words such as 'artificial intelligence', 'solar system planets', 'world war history' and 'machine learning' and returning the  query, the retrieved documents, and the similarity scores

In [12]:
example_queries = [
    'artifical intelligence',
    'solar system plants',
    'world war history',
    'machine learning'
]

In [17]:
for query in example_queries:
    results = retrieve(query, documents, embeddings, model, top_k=5, titles=titles)
    if not results:                          # ← guard against empty/None
        print('  No results returned.')
        continue
    for r in results:
        title_str = f" [{r['title']}]" if 'title' in r else ''
        print(f"  Rank {r['rank']} | Score: {r['score']:.4f}{title_str}")
        print(f"  {r['text'][:200]}...")
        print()

  Rank 1 | Score: 0.5378 [Artificial intelligence]
  Artificial intelligence (AI) is the ability of a computer program or a machine to think and learn. It is a field of study which tries to make computers "smart". They work on their own without being en...

  Rank 2 | Score: 0.3678 [Creativity]
  Creativity is the ability of a person or group to make something new and useful or valuable, or the process of making something new and useful or valuable. It happens in all areas of life - science, a...

  Rank 3 | Score: 0.3250 [Moral reasoning]
  Moral reasoning is a topic studied in psychology and in moral philosophy. It studies how people think about moral issues, problems, and questions. Psychologists who have studied it include Lawrence Ko...

  Rank 4 | Score: 0.3193 [Human science]
  Human science is the science of humans:  what makes them different from animals, and their limits, which tend to be the same as those of other animals.  Because human bodies are animal bodies, human s...


### Analysis of Retrieval Results

The system performs well when the query topic is clearly represented in the corpus. For *artificial intelligence*, the top result is correct (0.54), while lower-ranked results are weaker due to the limited corpus size. *Solar system planets* shows strong performance, retrieving Solar System, Venus, Planet, and Jupiter with solid similarity scores (0.40–0.52), with only a minor mismatch. *World war history* performs best, with highly relevant results such as World War (0.59), 1945 (0.52), and Cold War (0.47). In contrast, *machine learning* yields weaker scores (top score 0.26) because the selected corpus slice does not contain a dedicated ML article; related topics like Computer and Artificial Intelligence act as fallback matches.

Similarity scores above 0.50 indicate strong topical alignment, 0.30–0.50 reflect moderate relevance, and scores below 0.30 suggest the absence of a directly relevant article.

The main limitations are the small corpus size (2000 articles), embedding only the first 400 characters per article, and the absence of document chunking for finer retrieval. Additionally, the chosen model (all-MiniLM-L6-v2) prioritizes speed over depth; larger models such as E5-large or BGE would likely improve technical query performance. Finally, real-world systems rely on approximate nearest-neighbor search (e.g., FAISS) for scalability, whereas brute-force cosine similarity is suitable only for small datasets.